# ShelfRank: fine-tune bge-small on ESCI

Needs a GPU: **Runtime > Change runtime type > T4 GPU**. Then **Runtime > Run all** and allow
access to Google Drive. Everything that matters is kept in one Drive folder, about 2 GB while
training.

If the session ends, choose **Run all** again. Finished stages are skipped and training continues
from the last checkpoint. The last cell downloads `bge_small_esci.zip`.

The code cells are copies of the repository's tested modules. Nothing needs editing.

In [ ]:
# Every path and setting is here.
import os, subprocess, sys
from pathlib import Path

PACKAGES = ['transformers==4.45.2', 'onnx', 'onnxruntime']
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_WORK, DEFAULT_KEEP = "/content/shelfrank_ft", "/content/drive/MyDrive/shelfrank_ft"
elif Path("/kaggle").exists():
    DEFAULT_WORK, DEFAULT_KEEP = "/tmp/shelfrank_ft", "/kaggle/working/shelfrank_ft"
else:
    DEFAULT_WORK, DEFAULT_KEEP = "shelfrank_ft_work", "shelfrank_ft"
WORK = Path(os.environ.get("SHELFRANK_WORK", DEFAULT_WORK)).resolve()
KEEP = Path(os.environ.get("SHELFRANK_KEEP", DEFAULT_KEEP)).resolve()

SETTINGS = {'locale': 'us', 'split': 'train', 'n_held_out': 500, 'max_bullet_chars': 400, 'near_duplicate_jaccard': 0.75, 'seed': 20260}
TRAINING = {'batch_size': 64, 'learning_rate': 2e-05, 'max_query_tokens': 32, 'max_passage_tokens': 128, 'checkpoint_every': 500, 'max_minutes': 100, 'seed': 20260}
SOURCES = {'examples': {'url': 'https://media.githubusercontent.com/media/amazon-science/esci-data/7916cdf6ab75a462e77f20ab40428a10923998d5/shopping_queries_dataset/shopping_queries_dataset_examples.parquet', 'sha256': '4a735b693b4a424a6fc67f5be6e4c811495c488bbf66d02a602d308b2744263a'}, 'products': {'url': 'https://media.githubusercontent.com/media/amazon-science/esci-data/7916cdf6ab75a462e77f20ab40428a10923998d5/shopping_queries_dataset/shopping_queries_dataset_products.parquet', 'sha256': '25124442d064d64b26f74082d6fa09438d679efc0c183cf28d19064a2b65a265'}, 'wands_queries': {'url': 'https://raw.githubusercontent.com/wayfair/WANDS/3b74dcf4ba29ab8ff3e6a50b5b09fc627cb882b5/dataset/query.csv', 'sha256': '63b61660560fecc33ec490804c7e2b81402ee3e7c31a9cbb5e03736639f68e95'}}
MODEL = {'model': 'BAAI/bge-small-en-v1.5', 'revision': '5c38ec7c405ec4b44b94cc5a9bb96e735b38267a', 'query_prefix': 'Represent this sentence for searching relevant passages: '}
PROVENANCE = {'git_commit': '84a22bf6ca9e4f454c2f3ea4b7fded8f33ceb0b1', 'base_model': 'BAAI/bge-small-en-v1.5', 'base_revision': '5c38ec7c405ec4b44b94cc5a9bb96e735b38267a', 'code_sha256': {'shelfrank/text/normalize.py': '547f5fc5d8702035166a0ec1a72c2018bd5a73f66371356f52ac6d9fd15045ce', 'shelfrank/text/near_duplicates.py': '327589be5d32e67407a43d699c6b1d0d2dfc549ed2f64c49d70e1644ea33ff16', 'shelfrank/data/download.py': 'c5208ef25abede79441a0992a46bbbc1c77523f7dcf2f2b88499e0b134dd37ff', 'shelfrank/qu/seed_selection.py': '45c92fb25402f6778f3d0c78f1f5a34c1a456ce5bd47c5ab3aee9f159d63103d', 'shelfrank/dense/esci_data.py': '8b28634dd2f9560ad62c18598317b2008c7aa49ebfe57647590f29b16ed14742', 'shelfrank/dense/finetune.py': '03d889405bf80ec9693b24914dd3e03a65256a2b4956cc383e7aa84960e29734', 'shelfrank/dense/onnx_export.py': '2e17929166f8832eb5629b7efdbe801caaf58d13d4cad3937428bede56d3e659', 'scripts/finetune_dense.py': '88b835c490002eebfd88fbeee037042001ca80a1798c1ab97f649d1603f40c0b'}}

for folder in ("shelfrank/text", "shelfrank/data", "shelfrank/qu", "shelfrank/dense", "scripts"):
    (WORK / folder).mkdir(parents=True, exist_ok=True)
    (WORK / folder / "__init__.py").touch()
(WORK / "shelfrank" / "__init__.py").touch()
KEEP.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device", DEVICE, "| kept in", KEEP)
assert DEVICE == "cuda" or os.environ.get("SHELFRANK_ALLOW_CPU"), (
    "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Run all again.")

In [ ]:
%%writefile shelfrank/text/normalize.py
"""One text pipeline for queries and products, so that both sides always agree.

Every rule is a general property of English or of catalogue typography. None was derived from WANDS
queries.
"""

import re
import unicodedata

_PUNCTUATION = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"', "″": '"', "×": "x", "–": "-"})
# A quote mark or "in" counts as inches only straight after a number, since "in" alone is a
# preposition, and not before another number, since "3 in 1" is not a length.
_INCH = re.compile(r"(?<=\d)\s*(?:''|\"|inch(?:es)?\b|in\b(?!\s*\d)\.?)")
_FOOT = re.compile(r"(?<=\d)\s*(?:'(?!')|feet\b|foot\b|ft\b\.?)")
_POSSESSIVE = re.compile(r"(?<=[a-z])\s*'s\b")
_TOKEN = re.compile(r"\d+(?:\.\d+)?|[a-z]+")

STOPWORDS = frozenset({"a", "an", "and", "the", "of", "for", "with", "in", "on", "to", "by", "or"})

# British to American, limited to words that occur in home goods.
SPELLING = {
    "aluminium": "aluminum",
    "armour": "armor",
    "centre": "center",
    "centrepiece": "centerpiece",
    "colour": "color",
    "coloured": "colored",
    "fibre": "fiber",
    "grey": "gray",
    "jewellery": "jewelry",
    "metre": "meter",
    "mould": "mold",
    "moulding": "molding",
    "organiser": "organizer",
    "plough": "plow",
    "storey": "story",
}
IRREGULAR_PLURALS = {
    "children": "child",
    "knives": "knife",
    "leaves": "leaf",
    "loaves": "loaf",
    "men": "man",
    "mice": "mouse",
    "shelves": "shelf",
    "wolves": "wolf",
    "women": "woman",
}
# Singular already, although they end in a plural-looking s.
INVARIANT = frozenset({"bus", "canvas", "christmas", "gas", "lens", "plus", "series", "stainless"})


def normalize(text: str) -> str:
    """Lowercase, strip accents, unify quotes and spell out inch and foot marks."""
    folded = unicodedata.normalize("NFKD", text.translate(_PUNCTUATION).lower())
    plain = "".join(char for char in folded if not unicodedata.combining(char))
    plain = _POSSESSIVE.sub("", plain)
    plain = _INCH.sub(" inch ", plain)
    plain = _FOOT.sub(" ft ", plain)
    return re.sub(r"\s+", " ", plain).strip()


def singular(token: str) -> str:
    """Strip a regular plural ending.

    A conservative rule rather than a Porter stemmer: conflating "curtained" with "curtain" or
    "bedding" with "bed" would merge different product types.
    """
    if token in IRREGULAR_PLURALS:
        return IRREGULAR_PLURALS[token]
    if token in INVARIANT or len(token) <= 3 or not token.endswith("s"):
        return token
    if token.endswith(("ss", "us", "is")):
        return token
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith(("sses", "shes", "ches", "xes", "zes")):
        return token[:-2]
    return token[:-1]


def tokenize(text: object) -> list[str]:
    """Split into index terms; non-text (a missing cell) gives no terms.

    Hyphens split, so "mid-century" matches "mid century". Digits and letters split, so "54in" and
    "54 in" agree.
    """
    if not isinstance(text, str):
        return []
    tokens = []
    for raw in _TOKEN.findall(normalize(text)):
        token = singular(SPELLING.get(raw, raw))
        token = SPELLING.get(token, token)
        if token not in STOPWORDS:
            tokens.append(token)
    return tokens


In [ ]:
%%writefile shelfrank/text/near_duplicates.py
"""Find queries similar enough that putting them in different folds would leak labels."""

import re
from collections.abc import Mapping
from itertools import combinations

from shelfrank.text.normalize import tokenize

_TOKEN = re.compile(r"[a-z0-9]+")


def token_key(query: str) -> frozenset[str]:
    """Reduce a query to a word-order-free set with plural endings removed.

    Deliberately crude: this only has to over-group near-identical queries, not rank anything.
    """
    tokens = _TOKEN.findall(query.lower())
    return frozenset(t[:-1] if len(t) > 3 and t.endswith("s") else t for t in tokens)


def leakage_keys(query: str) -> tuple[frozenset[str], ...]:
    """Every key under which two queries may count as the same.

    The frozen folds use ``token_key``, which misses "vanity" against "vanities". Changing it would
    change the folds, so leakage checks add the full normaliser. No WANDS pair differs between them.
    """
    return tuple(dict.fromkeys(k for k in (token_key(query), frozenset(tokenize(query))) if k))


def jaccard(a: frozenset[str], b: frozenset[str]) -> float:
    """Set overlap; two empty sets are unrelated so that blank keys never group together."""
    union = a | b
    return len(a & b) / len(union) if union else 0.0


def near_duplicate_groups(queries: Mapping[int, str], threshold: float) -> list[list[int]]:
    """Return connected components (size two or more) of the similarity graph.

    Components rather than pairs: if A resembles B and B resembles C, all three must travel
    together even when A and C fall just under the threshold.
    """
    parent = {query_id: query_id for query_id in queries}

    def find(node: int) -> int:
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    keys = {query_id: token_key(text) for query_id, text in queries.items()}
    for a, b in combinations(sorted(keys), 2):
        if jaccard(keys[a], keys[b]) >= threshold:
            parent[find(b)] = find(a)

    groups: dict[int, list[int]] = {}
    for query_id in sorted(parent):
        groups.setdefault(find(query_id), []).append(query_id)
    return [members for members in groups.values() if len(members) > 1]


In [ ]:
%%writefile shelfrank/data/download.py
"""Fetch pinned files and refuse anything whose content differs from the recorded hash."""

import hashlib
import urllib.request
from collections.abc import Callable
from pathlib import Path

CHUNK = 1 << 20
Opener = Callable[[str, Path], None]


class ChecksumError(RuntimeError):
    """The downloaded bytes are not the bytes the results were computed from."""


def sha256_of(path: Path) -> str:
    """Hash in chunks so the 90 MB product file never has to fit in memory at once."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(CHUNK):
            digest.update(block)
    return digest.hexdigest()


def http_fetch(url: str, target: Path) -> None:
    """Stream ``url`` to ``target``."""
    if not url.startswith("https://"):
        raise ValueError(f"refusing a non-HTTPS download: {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "dataset-fetch/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response, target.open("wb") as out:
        while block := response.read(CHUNK):
            out.write(block)


def ensure_file(url: str, target: Path, sha256: str, fetch: Opener = http_fetch) -> bool:
    """Make ``target`` exist with the expected hash; return True if a download happened.

    The download goes to a temporary name first, so an interrupted run cannot leave a truncated
    file that a later run would mistake for the real one.
    """
    if target.exists() and sha256_of(target) == sha256:
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".part")
    try:
        fetch(url, partial)
        actual = sha256_of(partial)
        if actual != sha256:
            raise ChecksumError(f"{target.name}: expected {sha256}, got {actual}")
        partial.replace(target)
    finally:
        partial.unlink(missing_ok=True)
    return True


In [ ]:
%%writefile shelfrank/qu/seed_selection.py
"""Choose ESCI queries to label. Runs inside the Colab notebook, tested here.

Only pyarrow and the standard library are used, so the notebook needs no installation.
"""

import hashlib
import random
import re
from collections import Counter, defaultdict
from collections.abc import Collection, Iterable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from shelfrank.text.near_duplicates import jaccard, leakage_keys
from shelfrank.text.normalize import tokenize

_ASIN = re.compile(r"^b0[a-z0-9]{8}$")
NO_CLASS = "__no_class__"


@dataclass
class QueryEvidence:
    """How many of a query's Exact products matched each class."""

    text: str
    products: list[str] = field(default_factory=list)
    votes: Counter = field(default_factory=Counter)


def is_usable(query: str) -> bool:
    """Keep plain-text queries a shopper could type; drop product codes and other scripts."""
    words = query.split()
    letters = sum(char.isalpha() for char in query)
    if not query.isascii() or _ASIN.match(query.lower()):
        return False
    return 1 <= len(words) <= 10 and letters >= 3


def term_index(class_terms: Mapping[str, Iterable[str]]) -> dict[tuple[str, ...], str]:
    """Invert class terms for lookup."""
    return {tuple(term.split()): label for label, terms in class_terms.items() for term in terms}


def match_class(title: str, index: Mapping[tuple[str, ...], str], max_n: int = 4) -> str | None:
    """Class of the longest term in a title, earliest on a tie.

    Longest first, so "desk lamp" is read as a lamp and not as a desk. Earliest next, because
    marketplace titles put the product type before the search keywords.
    """
    tokens = tokenize(title)
    for size in range(min(max_n, len(tokens)), 0, -1):
        for start in range(len(tokens) - size + 1):
            label = index.get(tuple(tokens[start : start + size]))
            if label is not None:
                return label
    return None


def read_exact_pairs(examples: Path, locale: str, split: str) -> dict[str, QueryEvidence]:
    """Queries with their Exact products. The filter is pushed into the parquet scan."""
    import pyarrow.dataset as ds

    wanted = (
        (ds.field("product_locale") == locale)
        & (ds.field("esci_label") == "E")
        & (ds.field("split") == split)
    )
    queries: dict[str, QueryEvidence] = {}
    scanner = ds.dataset(examples).scanner(
        columns=["query_id", "query", "product_id"], filter=wanted
    )
    for batch in scanner.to_batches():
        columns = [batch.column(i).to_pylist() for i in range(3)]
        for query_id, text, product_id in zip(*columns, strict=True):
            evidence = queries.setdefault(str(query_id), QueryEvidence(str(text).strip()))
            evidence.products.append(product_id)
    return queries


def add_votes(
    queries: Mapping[str, QueryEvidence],
    products: Path,
    locale: str,
    index: Mapping[tuple[str, ...], str],
    batch_size: int = 20_000,
) -> int:
    """Stream product titles and count class matches per query; return the titles read.

    Batches are dropped after use and only the two needed columns are read, so memory stays flat
    however large the products file is.
    """
    import pyarrow.dataset as ds

    wanted: dict[str, list[str]] = defaultdict(list)
    for query_id, evidence in queries.items():
        for product_id in evidence.products:
            wanted[product_id].append(query_id)
    read = 0
    scanner = ds.dataset(products).scanner(
        columns=["product_id", "product_title"],
        filter=ds.field("product_locale") == locale,
        batch_size=batch_size,
        batch_readahead=2,
        fragment_readahead=1,
    )
    for batch in scanner.to_batches():
        ids, titles = batch.column(0).to_pylist(), batch.column(1).to_pylist()
        for product_id, title in zip(ids, titles, strict=True):
            if product_id in wanted and title:
                read += 1
                label = match_class(title, index)
                if label is not None:
                    for query_id in wanted[product_id]:
                        queries[query_id].votes[label] += 1
    return read


def assign_pools(
    queries: Mapping[str, QueryEvidence], min_votes: int, min_share: float
) -> dict[str, list[str]]:
    """Group query ids by their majority class, or by having no class evidence at all.

    Queries with weak evidence join neither pool. The teacher decides every label, so this only has
    to raise the share of home-goods queries in the sample.
    """
    pools: dict[str, list[str]] = defaultdict(list)
    for query_id, evidence in sorted(queries.items()):
        if not evidence.votes:
            if len(evidence.products) >= min_votes:
                pools[NO_CLASS].append(query_id)
            continue
        label, count = min(evidence.votes.items(), key=lambda item: (-item[1], item[0]))
        if count >= min_votes and count / len(evidence.products) >= min_share:
            pools[label].append(query_id)
    return dict(pools)


def spread(pools: Mapping[str, Sequence[str]], total: int, rng: random.Random) -> list[str]:
    """Take one query per class in turn until ``total``, so rare classes are represented."""
    shuffled = {label: rng.sample(list(ids), len(ids)) for label, ids in sorted(pools.items())}
    chosen: list[str] = []
    while len(chosen) < total and any(shuffled.values()):
        for label in sorted(shuffled):
            if shuffled[label] and len(chosen) < total:
                chosen.append(shuffled[label].pop())
    return chosen


def resembles(query: str, keys: Collection[frozenset[str]], threshold: float) -> bool:
    """Whether a query is the same as, or nearly the same as, any protected query."""
    return any(jaccard(own, key) >= threshold for own in leakage_keys(query) for key in keys)


def filter_queries(
    queries: Mapping[str, QueryEvidence], protected: Iterable[str], threshold: float
) -> tuple[dict[str, QueryEvidence], Counter]:
    """Drop unusable queries, repeats, and anything that resembles a protected query."""
    by_token: dict[str, set[frozenset[str]]] = defaultdict(set)
    for text in protected:
        for key in leakage_keys(text):
            for token in key:
                by_token[token].add(key)
    kept: dict[str, QueryEvidence] = {}
    seen: set[frozenset[str]] = set()
    dropped: Counter = Counter()
    for query_id, evidence in sorted(queries.items()):
        keys = leakage_keys(evidence.text)
        tokens = frozenset().union(*keys)
        nearby = set().union(*(by_token[t] for t in tokens if t in by_token))
        if not keys or not is_usable(evidence.text):
            dropped["unusable"] += 1
        elif seen.intersection(keys):
            dropped["repeat"] += 1
        elif resembles(evidence.text, nearby, threshold):
            dropped["resembles a protected query"] += 1
        else:
            seen.update(keys)
            kept[query_id] = evidence
    return kept, dropped


def select(
    pools: Mapping[str, Sequence[str]],
    n_seed: int,
    n_dev: int,
    home_share: float,
    seed: int,
) -> tuple[list[str], list[str]]:
    """Draw disjoint seed and prompt-development samples from the same mixture."""
    rng = random.Random(seed)
    total = n_seed + n_dev
    n_home = round(total * home_share)
    home = spread({k: v for k, v in pools.items() if k != NO_CLASS}, n_home, rng)
    outside = list(pools.get(NO_CLASS, []))
    chosen = home + rng.sample(outside, min(total - n_home, len(outside)))
    if len(chosen) < total:
        raise ValueError(f"only {len(chosen)} of {total} queries available: loosen the settings")
    rng.shuffle(chosen)
    return chosen[n_dev:], chosen[:n_dev]


def write_outputs(
    directory: Path, seed_texts: Sequence[str], dev_texts: Sequence[str]
) -> dict[str, Any]:
    """Write one query per line and return the hashes for the manifest."""
    directory.mkdir(parents=True, exist_ok=True)
    out = {}
    for name, texts in (("esci_seed_queries.txt", seed_texts), ("esci_prompt_dev.txt", dev_texts)):
        body = "".join(" ".join(text.split()) + "\n" for text in texts).encode("utf-8")
        (directory / name).write_bytes(body)
        out[name] = {"lines": len(texts), "sha256": hashlib.sha256(body).hexdigest()}
    return out


In [ ]:
%%writefile shelfrank/dense/esci_data.py
"""Build ESCI training triplets. Runs in Colab, tested here on synthetic files.

Only pyarrow and the standard library are used.
"""

import json
import random
from collections import defaultdict
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

from shelfrank.qu.seed_selection import QueryEvidence, filter_queries

# Gains of the ESCI paper, used only for the held-out ESCI check.
ESCI_GAINS = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}
Judged = dict[str, dict[str, str]]


@dataclass
class TrainingData:
    """Queries, passages and the judgments that connect them."""

    queries: dict[str, str] = field(default_factory=dict)
    passages: dict[str, str] = field(default_factory=dict)
    train: Judged = field(default_factory=dict)
    held_out: Judged = field(default_factory=dict)
    report: dict[str, Any] = field(default_factory=dict)

    def save(self, path: Path) -> None:
        """Write to one file, under a temporary name first so that a half file is never read."""
        partial = path.with_suffix(".partial")
        partial.write_text(json.dumps(asdict(self)), encoding="utf-8")
        partial.replace(path)

    @staticmethod
    def load(path: Path) -> "TrainingData":
        """Read a saved file."""
        return TrainingData(**json.loads(path.read_text(encoding="utf-8")))


def read_judgments(examples: Path, locale: str, split: str) -> tuple[dict[str, str], Judged]:
    """Queries and judgments of the small version, with the filter pushed into the scan."""
    import pyarrow.dataset as ds

    wanted = (
        (ds.field("product_locale") == locale)
        & (ds.field("split") == split)
        & (ds.field("small_version") == 1)
    )
    columns = ["query_id", "query", "product_id", "esci_label"]
    queries: dict[str, str] = {}
    judged: Judged = defaultdict(dict)
    for batch in ds.dataset(examples).scanner(columns=columns, filter=wanted).to_batches():
        values = [batch.column(i).to_pylist() for i in range(4)]
        for query_id, text, product, label in zip(*values, strict=True):
            queries[str(query_id)] = str(text).strip()
            judged[str(query_id)][product] = label
    return queries, dict(judged)


def passage(row: Mapping[str, object], max_chars: int) -> str:
    """Title, brand, colour and the start of the bullet points, most specific first."""
    parts = [row.get("product_title"), row.get("product_brand"), row.get("product_color")]
    bullets = row.get("product_bullet_point")
    if isinstance(bullets, str):
        parts.append(bullets[:max_chars])
    return " . ".join(" ".join(p.split()) for p in parts if isinstance(p, str) and p.strip())


def read_passages(
    products: Path, locale: str, wanted: set[str], max_chars: int, batch_size: int = 20_000
) -> dict[str, str]:
    """Stream the product file and keep only judged products."""
    import pyarrow.dataset as ds

    columns = ["product_title", "product_brand", "product_color", "product_bullet_point"]
    out: dict[str, str] = {}
    scanner = ds.dataset(products).scanner(
        columns=["product_id", *columns],
        filter=ds.field("product_locale") == locale,
        batch_size=batch_size,
        batch_readahead=2,
        fragment_readahead=1,
    )
    for batch in scanner.to_batches():
        for row in batch.to_pylist():
            if row["product_id"] in wanted:
                text = passage(row, max_chars)
                if text:
                    out[row["product_id"]] = text
    return out


def prepare(
    examples: Path,
    products: Path,
    protected: Iterable[str],
    settings: Mapping[str, Any],
) -> TrainingData:
    """Read ESCI, drop look-alikes of protected queries, and hold queries out."""
    queries, judged = read_judgments(examples, settings["locale"], settings["split"])
    evidence = {q: QueryEvidence(text) for q, text in queries.items()}
    kept, dropped = filter_queries(evidence, protected, settings["near_duplicate_jaccard"])
    ids = sorted(kept)
    random.Random(settings["seed"]).shuffle(ids)
    held_ids, train_ids = ids[: settings["n_held_out"]], ids[settings["n_held_out"] :]
    # A query with no Exact product gives no positive.
    train_ids = [q for q in train_ids if "E" in judged[q].values()]
    wanted = {p for q in (*held_ids, *train_ids) for p in judged[q]}
    passages = read_passages(products, settings["locale"], wanted, settings["max_bullet_chars"])

    def with_text(ids: Sequence[str]) -> Judged:
        return {q: {p: v for p, v in judged[q].items() if p in passages} for q in ids}

    data = TrainingData(
        {q: queries[q] for q in (*held_ids, *train_ids)},
        passages,
        with_text(train_ids),
        with_text(held_ids),
    )
    data.report = {
        "queries_read": len(queries),
        "dropped": dict(sorted(dropped.items())),
        "train_queries": len(data.train),
        "held_out_queries": len(data.held_out),
        "passages": len(passages),
        "positives": sum(list(v.values()).count("E") for v in data.train.values()),
    }
    return data


@dataclass(frozen=True)
class Triplet:
    """A query, an Exact product and an Irrelevant product judged for the same query."""

    query: str
    positive: str
    negative: str
    hard: bool


def make_triplets(judged: Judged, passages: Mapping[str, str], seed: int) -> list[list[Triplet]]:
    """Triplets grouped in rounds; a round holds at most one triplet per query.

    A query with no Irrelevant judgment gets a random product, recorded as not hard.
    """
    rng = random.Random(seed)
    everything = sorted(passages)
    rounds: dict[int, list[Triplet]] = defaultdict(list)
    for query, labels in sorted(judged.items()):
        positives = sorted(p for p, v in labels.items() if v == "E")
        negatives = sorted(p for p, v in labels.items() if v == "I")
        rng.shuffle(positives)
        rng.shuffle(negatives)
        for i, positive in enumerate(positives):
            if negatives:
                negative, hard = negatives[i % len(negatives)], True
            else:
                negative, hard = rng.choice(everything), False
            rounds[i].append(Triplet(query, positive, negative, hard))
    for group in rounds.values():
        rng.shuffle(group)
    return [rounds[i] for i in sorted(rounds)]


def make_batches(rounds: Sequence[Sequence[Triplet]], size: int, seed: int) -> list[list[Triplet]]:
    """Batches with distinct queries, in random order. Short leftovers are dropped.

    Distinct queries matter: two positives of one query in a batch would be each other's negative.
    """
    batches = [
        list(group[start : start + size])
        for group in rounds
        for start in range(0, len(group) - size + 1, size)
    ]
    random.Random(seed).shuffle(batches)
    return batches


In [ ]:
%%writefile shelfrank/dense/finetune.py
"""Fine-tune a bi-encoder with in-batch and hard negatives, resumable from checkpoints.

A plain PyTorch loop and not a trainer library: the loss is ten lines, resuming is exact because
every step sets its own random seed, and it does not depend on which library versions Colab ships.
"""

import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import torch
from torch import nn

from shelfrank.dense.esci_data import ESCI_GAINS, Judged, TrainingData, Triplet

Log = Callable[[str], None]


@dataclass(frozen=True)
class TrainConfig:
    """Every training setting, written to the model card."""

    batch_size: int = 64
    learning_rate: float = 2e-5
    warmup_share: float = 0.1
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    scale: float = 20.0
    max_query_tokens: int = 32
    max_passage_tokens: int = 128
    checkpoint_every: int = 500
    log_every: int = 50
    max_minutes: float = 100.0
    seed: int = 20260


def embed(model: nn.Module, batch: Mapping[str, torch.Tensor]) -> torch.Tensor:
    """Unit vectors from the first token, which is how bge models pool."""
    hidden = model(**batch).last_hidden_state[:, 0]
    return nn.functional.normalize(hidden.float(), dim=-1)


def contrastive_loss(
    queries: torch.Tensor,
    positives: torch.Tensor,
    negatives: torch.Tensor,
    blocked: torch.Tensor,
    scale: float,
) -> torch.Tensor:
    """Multiple-negatives ranking loss over in-batch positives and all hard negatives.

    ``blocked`` marks candidates known to be Exact for a query other than its own positive. Scoring
    them as negatives would teach the model something false.
    """
    scores = scale * queries @ torch.cat([positives, negatives]).T
    scores = scores.masked_fill(blocked, float("-inf"))
    return nn.functional.cross_entropy(scores, torch.arange(len(queries), device=scores.device))


def blocked_candidates(batch: Sequence[Triplet], judged: Judged) -> torch.Tensor:
    """True where a candidate is Exact for the row's query and is not the row's own positive."""
    candidates = [t.positive for t in batch] + [t.negative for t in batch]
    out = torch.zeros((len(batch), len(candidates)), dtype=torch.bool)
    for i, triplet in enumerate(batch):
        exact = judged[triplet.query]
        for j, product in enumerate(candidates):
            out[i, j] = j != i and exact.get(product) == "E"
    return out


def learning_rate_factor(step: int, total: int, warmup_share: float) -> float:
    """Linear warm-up, then linear decay to 0."""
    warmup = max(1, round(total * warmup_share))
    if step < warmup:
        return (step + 1) / warmup
    return max(0.0, (total - step) / max(1, total - warmup))


class Checkpoints:
    """Two alternating slots, so a crash while writing never destroys the only copy.

    Files are overwritten and not deleted, because deleted files on Google Drive go to the bin and
    still count against the quota.
    """

    def __init__(self, directory: Path, identity: Mapping[str, Any]) -> None:
        """Use ``directory``; ``identity`` describes the run the checkpoints belong to."""
        self.directory = directory
        self.identity = dict(identity)
        directory.mkdir(parents=True, exist_ok=True)

    def _slots(self) -> list[tuple[int, Path]]:
        found = []
        for name in ("a", "b"):
            note = self.directory / f"slot_{name}.json"
            if note.exists():
                try:
                    info = json.loads(note.read_text(encoding="utf-8"))
                except json.JSONDecodeError:
                    continue
                if info.get("complete"):
                    if info["identity"] != self.identity:
                        raise ValueError(
                            f"{self.directory} holds a run with other settings or data. "
                            "Use another folder, or delete it to start again."
                        )
                    found.append((int(info["step"]), self.directory / f"slot_{name}.pt"))
        return sorted(found)

    def save(self, step: int, state: dict[str, Any]) -> None:
        """Write the older slot and mark it complete last."""
        slots = self._slots()
        used = [path.stem[-1] for _, path in slots]
        name = next((n for n in ("a", "b") if n not in used), used[0] if used else "a")
        note = self.directory / f"slot_{name}.json"
        body = {"step": step, "identity": self.identity}
        note.write_text(json.dumps(body | {"complete": False}), encoding="utf-8")
        torch.save(state, self.directory / f"slot_{name}.pt")
        note.write_text(json.dumps(body | {"complete": True}), encoding="utf-8")

    def latest(self) -> dict[str, Any] | None:
        """Newest complete checkpoint."""
        slots = self._slots()
        if not slots:
            return None
        return torch.load(slots[-1][1], map_location="cpu", weights_only=True)


class RunLog:
    """What happened in each session, kept on Drive because a closed tab loses the printed output.

    Written at every logging step, so a session that dies still shows how long it ran. The losses
    after a checkpoint are dropped on resuming, because those steps run again.
    """

    def __init__(self, path: Path) -> None:
        """Continue the log at ``path``, or start one."""
        self.path = path
        self.body: dict[str, Any] = {"sessions": [], "resumes": [], "loss_per_step": []}
        if path.exists():
            self.body = json.loads(path.read_text(encoding="utf-8"))

    @staticmethod
    def _now() -> str:
        return datetime.now(UTC).isoformat(timespec="seconds")

    def start(self, step: int) -> None:
        """Open a session at ``step``. A step above 0 is a resume."""
        self.started = time.monotonic()
        self.body["loss_per_step"] = self.body["loss_per_step"][:step]
        if step:
            self.body["resumes"].append({"time": self._now(), "message": f"resumed at step {step}"})
        session = {"started": self._now(), "first_step": step, "last_step": step, "seconds": 0.0}
        self.body["sessions"].append(session | {"ended": "still running or lost"})
        self.write(step)

    def write(self, step: int, ended: str | None = None) -> None:
        """Save the state so far, replacing the file in one move."""
        session = self.body["sessions"][-1]
        session["last_step"] = step
        session["seconds"] = round(time.monotonic() - self.started, 1)
        if ended:
            session["ended"] = ended
        self.path.parent.mkdir(parents=True, exist_ok=True)
        partial = self.path.with_suffix(".partial")
        partial.write_text(json.dumps(self.body, indent=1) + "\n", encoding="utf-8")
        partial.replace(self.path)


def train(
    model: nn.Module,
    tokenizer: Any,  # noqa: ANN401 - transformers tokenizers share no base type worth importing
    data: TrainingData,
    batches: Sequence[Sequence[Triplet]],
    config: TrainConfig,
    checkpoints: Checkpoints,
    query_prefix: str,
    device: str,
    log: Log = print,
    record: RunLog | None = None,
) -> dict[str, Any]:
    """Run until the epoch ends or the time limit is reached. Call again to continue."""
    started = time.monotonic()
    total = len(batches)
    model.to(device)
    decay = [p for p in model.parameters() if p.ndim > 1]
    plain = [p for p in model.parameters() if p.ndim <= 1]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": config.weight_decay},
            {"params": plain, "weight_decay": 0.0},
        ],
        lr=config.learning_rate,
    )
    half = device == "cuda"
    scaler = torch.amp.GradScaler(device, enabled=half)
    step, history = 0, []
    state = checkpoints.latest()
    if state is not None:
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        step, history = int(state["step"]), list(state["history"])
        log(f"resumed at step {step} of {total}")

    if record and step < total:
        record.start(step)

    def save() -> None:
        checkpoints.save(
            step,
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "step": step,
                "history": history,
            },
        )

    def tokens(texts: list[str], limit: int) -> dict[str, torch.Tensor]:
        encoded = tokenizer(
            texts, padding=True, truncation=True, max_length=limit, return_tensors="pt"
        )
        return {k: v.to(device) for k, v in encoded.items()}

    model.train()
    running: list[float] = []
    first = step
    while step < total:
        if (time.monotonic() - started) / 60 >= config.max_minutes:
            save()
            if record:
                record.write(step, "time limit")
            log(f"time limit reached at step {step} of {total}; run the cell again to continue")
            break
        batch = batches[step]
        # Seeding each step makes a resumed run identical to an uninterrupted one.
        torch.manual_seed(config.seed + step)
        for group in optimizer.param_groups:
            group["lr"] = config.learning_rate * learning_rate_factor(
                step, total, config.warmup_share
            )
        queries = tokens(
            [query_prefix + data.queries[t.query] for t in batch], config.max_query_tokens
        )
        passages = tokens(
            [data.passages[t.positive] for t in batch] + [data.passages[t.negative] for t in batch],
            config.max_passage_tokens,
        )
        with torch.autocast(device, enabled=half):
            q, p = embed(model, queries), embed(model, passages)
        loss = contrastive_loss(
            q,
            p[: len(batch)],
            p[len(batch) :],
            blocked_candidates(batch, data.train).to(device),
            config.scale,
        )
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        step += 1
        running.append(float(loss.detach()))
        if record:
            record.body["loss_per_step"].append(running[-1])
        if step % config.log_every == 0 or step == total:
            history.append([step, sum(running) / len(running)])
            minutes = (time.monotonic() - started) / 60
            left = minutes / max(1, step - first) * (total - step)
            log(
                f"step {step}/{total}  loss {history[-1][1]:.4f}  "
                f"{minutes:.0f} min, about {left:.0f} min left"
            )
            running = []
            if record:
                record.write(step, "finished" if step == total else None)
        if step % config.checkpoint_every == 0 or step == total:
            save()
    return {"step": step, "total": total, "finished": step >= total, "history": history}


@torch.no_grad()
def encode(
    model: nn.Module,
    tokenizer: Any,  # noqa: ANN401
    texts: Sequence[str],
    limit: int,
    device: str,
    batch_size: int = 128,
) -> torch.Tensor:
    """Unit vectors on the CPU, in input order."""
    model.to(device).eval()
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    out: list[torch.Tensor | None] = [None] * len(texts)
    for start in range(0, len(order), batch_size):
        chunk = order[start : start + batch_size]
        encoded = tokenizer(
            [texts[i] for i in chunk],
            padding=True,
            truncation=True,
            max_length=limit,
            return_tensors="pt",
        )
        vectors = embed(model, {k: v.to(device) for k, v in encoded.items()}).cpu()
        for i, vector in zip(chunk, vectors, strict=True):
            out[i] = vector
    return torch.stack([v for v in out if v is not None])


def ndcg(gains: Sequence[float]) -> float:
    """Full-list nDCG of gains in ranked order."""
    best = sum(g / math.log2(i + 2) for i, g in enumerate(sorted(gains, reverse=True)))
    got = sum(g / math.log2(i + 2) for i, g in enumerate(gains))
    return got / best if best else 0.0


def held_out_scores(
    model: nn.Module,
    tokenizer: Any,  # noqa: ANN401
    data: TrainingData,
    config: TrainConfig,
    query_prefix: str,
    device: str,
) -> dict[str, float]:
    """NDCG per held-out ESCI query over its judged products. Queries with no Exact are skipped.

    Per query and not only the mean, so that a paired interval can be computed later.
    """
    ids = sorted(q for q, v in data.held_out.items() if "E" in v.values())
    products = sorted({p for q in ids for p in data.held_out[q]})
    row = {p: i for i, p in enumerate(products)}
    texts = [query_prefix + data.queries[q] for q in ids]
    queries = encode(model, tokenizer, texts, config.max_query_tokens, device)
    passages = [data.passages[p] for p in products]
    vectors = encode(model, tokenizer, passages, config.max_passage_tokens, device)
    scores = {}
    for q, vector in zip(ids, queries, strict=True):
        judged = sorted(data.held_out[q])
        values = vectors[[row[p] for p in judged]] @ vector
        ranked = sorted(zip(values.tolist(), judged, strict=True), key=lambda x: (-x[0], x[1]))
        scores[q] = ndcg([ESCI_GAINS[data.held_out[q][p]] for _, p in ranked])
    return scores


In [ ]:
%%writefile shelfrank/dense/onnx_export.py
"""Export a bi-encoder to an int8 ONNX file for CPU serving, and check it against PyTorch."""

from collections.abc import Sequence
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch import nn

from shelfrank.dense.finetune import embed

INPUTS = ("input_ids", "attention_mask", "token_type_ids")
OPSET = 14


class _Encoder(nn.Module):
    """Takes plain tensors and returns the finished vector, so serving needs no pooling code."""

    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
    ) -> torch.Tensor:
        return embed(
            self.model,
            {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "token_type_ids": token_type_ids,
            },
        )


def export_fp32(model: nn.Module, tokenizer: Any, path: Path) -> None:  # noqa: ANN401
    """Trace with variable batch and sequence length."""
    model.to("cpu").eval()
    sample = tokenizer(["a lamp", "a red velvet sofa"], padding=True, return_tensors="pt")
    path.parent.mkdir(parents=True, exist_ok=True)
    extra: dict[str, Any] = {}
    if "dynamo" in torch.onnx.export.__code__.co_varnames:
        extra["dynamo"] = False
    with torch.no_grad():
        torch.onnx.export(
            _Encoder(model),
            tuple(sample[name] for name in INPUTS),
            str(path),
            input_names=list(INPUTS),
            output_names=["embedding"],
            dynamic_axes={
                **{name: {0: "batch", 1: "tokens"} for name in INPUTS},
                "embedding": {0: "batch"},
            },
            opset_version=OPSET,
            **extra,
        )


def quantize(source: Path, target: Path) -> None:
    """Dynamic int8 quantisation of the weights."""
    from onnxruntime.quantization import QuantType, quantize_dynamic

    quantize_dynamic(str(source), str(target), weight_type=QuantType.QInt8)


def onnx_encode(path: Path, tokenizer: Any, texts: Sequence[str], limit: int) -> np.ndarray:  # noqa: ANN401
    """Encode one text at a time, as serving does."""
    import onnxruntime as ort

    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    out = []
    for text in texts:
        encoded = tokenizer([text], truncation=True, max_length=limit, return_tensors="np")
        feed = {name: encoded[name].astype(np.int64) for name in INPUTS}
        out.append(session.run(["embedding"], feed)[0][0])
    return np.stack(out)


def agreement(reference: np.ndarray, other: np.ndarray) -> dict[str, float]:
    """Cosine similarity between two sets of unit vectors."""
    cosine = (reference * other).sum(axis=1) / np.linalg.norm(other, axis=1)
    return {"mean": float(cosine.mean()), "min": float(cosine.min())}


In [ ]:
%%writefile scripts/finetune_dense.py
"""Fine-tune the dense retriever on ESCI. Every stage can be repeated, and a repeat continues.

The notebook notebooks/finetune_bge_esci.ipynb calls these stages on a Colab GPU. ESCI is never
downloaded on the development laptop.
"""

import csv
import hashlib
import json
import platform
import shutil
import time
from collections.abc import Callable, Mapping
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch

from shelfrank.data.download import ensure_file
from shelfrank.dense.esci_data import TrainingData, make_batches, make_triplets, prepare
from shelfrank.dense.finetune import (
    Checkpoints,
    RunLog,
    TrainConfig,
    encode,
    held_out_scores,
    train,
)
from shelfrank.dense.onnx_export import agreement, export_fp32, onnx_encode, quantize

Log = Callable[[str], None]
MODEL_FILES = (
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "vocab.txt",
    "modules.json",
    "sentence_bert_config.json",
    "config_sentence_transformers.json",
    "1_Pooling/config.json",
)
# Invented, so that vectors made on the GPU machine can be compared with vectors made elsewhere.
REFERENCE_TEXTS = (
    "tangerine ukulele case with zipper",
    "two person kayak paddle set",
    "left handed fountain pen purple ink",
    "stainless steel bicycle bell",
    "gluten free sourdough starter kit",
    "waterproof hiking gaiters size medium",
    "usb c to ethernet adapter gigabit",
    "wooden chess pieces weighted no board",
    "toddler rain boots with dinosaurs",
    "replacement strap for a diving watch",
    "noise cancelling earbuds for running",
    "telescopic fishing rod saltwater",
    "dog harness reflective extra small",
    "refillable deodorant cedar scent",
    "mechanical keyboard with quiet switches",
    "Ukulele . Acme . tangerine . Concert size with a padded carrying bag",
    "Kayak paddle . Acme . Two pieces that lock together, 220 cm, fibreglass",
    "Fountain pen . Acme . purple . Fine nib for left handed writers, two cartridges",
    "Bicycle bell . Acme . silver . Fits handlebars from 22 to 32 mm",
    "Keyboard . Acme . black . 87 keys, hot swappable, quiet linear switches",
)


@dataclass(frozen=True)
class Paths:
    """Where things live. ``work`` is fast and lost with the session; ``keep`` survives it."""

    work: Path
    keep: Path

    @property
    def prepared(self) -> Path:
        """Triplet data, kept so that a new session skips the 1 GB product file."""
        return self.keep / "training_data.json"

    @property
    def checkpoints(self) -> Path:
        """Training state."""
        return self.keep / "checkpoints"

    @property
    def results(self) -> Path:
        """Measurements so far."""
        return self.keep / "finetune_results.json"

    @property
    def run_log(self) -> Path:
        """Sessions, resumes and every step's loss."""
        return self.keep / "run_log.json"

    @property
    def output(self) -> Path:
        """The finished package."""
        return self.keep / "package"


def read_results(paths: Paths) -> dict[str, Any]:
    """Measurements from earlier stages."""
    if not paths.results.exists():
        return {}
    return json.loads(paths.results.read_text(encoding="utf-8"))


def write_results(paths: Paths, **values: Any) -> None:  # noqa: ANN401
    """Add measurements."""
    body = json.dumps(read_results(paths) | values, indent=1)
    paths.results.write_text(body + "\n", encoding="utf-8")


def stage_data(
    paths: Paths, sources: Mapping[str, Any], settings: Mapping[str, Any], log: Log = print
) -> TrainingData:
    """Download, filter and save the training data, or load what an earlier run saved."""
    paths.keep.mkdir(parents=True, exist_ok=True)
    if paths.prepared.exists():
        log("training data: loaded from the kept folder")
        return TrainingData.load(paths.prepared)
    files = {name: paths.work / f"{name}.file" for name in sources}
    for name, spec in sources.items():
        files[name].parent.mkdir(parents=True, exist_ok=True)
        ensure_file(spec["url"], files[name], spec["sha256"])
        log(f"downloaded {name}")
    # WANDS queries are read by code only, to remove look-alikes. Never printed or saved.
    with files["wands_queries"].open(encoding="utf-8", newline="") as handle:
        protected = [row["query"] for row in csv.DictReader(handle, delimiter="\t")]
    data = prepare(files["examples"], files["products"], protected, settings)
    data.save(paths.prepared)
    write_results(paths, data=data.report, settings=dict(settings))
    log(json.dumps(data.report, indent=1))
    return data


def load_base(directory: Path) -> tuple[Any, Any]:
    """Base model and tokenizer from a local folder."""
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(directory)
    model = AutoModel.from_pretrained(directory, add_pooling_layer=False)
    return model, tokenizer


def download_base(work: Path, model: str, revision: str) -> Path:
    """Fetch the pinned base model."""
    from huggingface_hub import snapshot_download

    target = work / "base"
    allow = [*MODEL_FILES, "model.safetensors"]
    snapshot_download(model, revision=revision, local_dir=target, allow_patterns=allow)
    return target


def data_fingerprint(data: TrainingData) -> str:
    """Hash of the training data, so checkpoints never continue on other data."""
    body = json.dumps(asdict(data), sort_keys=True)
    return hashlib.sha256(body.encode("utf-8")).hexdigest()


def stage_train(
    paths: Paths,
    data: TrainingData,
    base: Path,
    config: TrainConfig,
    query_prefix: str,
    device: str,
    log: Log = print,
) -> dict[str, Any]:
    """Measure the base model once, then train or continue training."""
    model, tokenizer = load_base(base)
    if "held_out_ndcg_before" not in read_results(paths):
        scores = held_out_scores(model, tokenizer, data, config, query_prefix, device)
        before = sum(scores.values()) / len(scores)
        write_results(paths, held_out_ndcg_before=before, held_out_before_per_query=scores)
        log(f"held-out ESCI nDCG of the base model: {before:.4f}")
    batches = make_batches(
        make_triplets(data.train, data.passages, config.seed), config.batch_size, config.seed
    )
    if not batches:
        raise ValueError("not enough queries for one batch")
    hard = [t.hard for batch in batches for t in batch]
    # The time limit and the logging interval do not change the result, so they may change.
    identity = {
        **{k: v for k, v in asdict(config).items() if k not in ("max_minutes", "log_every")},
        "data": data_fingerprint(data),
        "query_prefix": query_prefix,
    }
    checkpoints = Checkpoints(paths.checkpoints, identity)
    record = RunLog(paths.run_log)
    out = train(
        model, tokenizer, data, batches, config, checkpoints, query_prefix, device, log, record
    )
    write_results(
        paths,
        train=out,
        config=asdict(config),
        triplets=len(hard),
        hard_negative_share=sum(hard) / len(hard),
    )
    return out


def trained_model(paths: Paths, base: Path) -> tuple[Any, Any]:
    """Base architecture with the newest trained weights."""
    results = read_results(paths)
    if not results.get("train", {}).get("finished"):
        raise RuntimeError("training has not finished: run the training cell again")
    model, tokenizer = load_base(base)
    newest = None
    for note in paths.checkpoints.glob("slot_*.json"):
        info = json.loads(note.read_text(encoding="utf-8"))
        if info["complete"] and (newest is None or info["step"] > newest[0]):
            newest = (info["step"], note.with_suffix(".pt"))
    assert newest is not None
    state = torch.load(newest[1], map_location="cpu", weights_only=True)
    model.load_state_dict(state["model"])
    return model, tokenizer


def stage_evaluate(
    paths: Paths,
    data: TrainingData,
    base: Path,
    config: TrainConfig,
    query_prefix: str,
    device: str,
    log: Log = print,
) -> None:
    """Held-out ESCI nDCG of the trained model."""
    model, tokenizer = trained_model(paths, base)
    scores = held_out_scores(model, tokenizer, data, config, query_prefix, device)
    after = sum(scores.values()) / len(scores)
    write_results(paths, held_out_ndcg_after=after, held_out_after_per_query=scores)
    log(f"held-out ESCI nDCG: {read_results(paths)['held_out_ndcg_before']:.4f} -> {after:.4f}")


def save_model(model: Any, base: Path, target: Path) -> None:  # noqa: ANN401
    """Write weights with the base model's own configuration files.

    Copying them keeps the folder readable by the older library versions used locally.
    """
    from safetensors.torch import save_file

    target.mkdir(parents=True, exist_ok=True)
    for name in MODEL_FILES:
        if (base / name).exists():
            (target / name).parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(base / name, target / name)
    weights = {k: v.detach().cpu().float().contiguous() for k, v in model.state_dict().items()}
    save_file(weights, str(target / "model.safetensors"), metadata={"format": "pt"})


def export_int8(
    model: Any,  # noqa: ANN401
    tokenizer: Any,  # noqa: ANN401
    target: Path,
    work: Path,
    texts: list[str],
    limit: int,
) -> dict[str, Any]:
    """Write model_int8.onnx and measure how far it is from PyTorch."""
    fp32 = work / f"{target.name}_fp32.onnx"
    export_fp32(model, tokenizer, fp32)
    out = target / "model_int8.onnx"
    out.parent.mkdir(parents=True, exist_ok=True)
    quantize(fp32, out)
    reference = encode(model, tokenizer, texts, limit, "cpu").numpy()
    started = time.perf_counter()
    vectors = onnx_encode(out, tokenizer, texts, limit)
    return {
        "megabytes": out.stat().st_size / 1e6,
        "cosine_to_pytorch": agreement(reference, vectors),
        "ms_per_text_colab_cpu": 1000 * (time.perf_counter() - started) / len(texts),
    }


def stage_package(
    paths: Paths,
    data: TrainingData,
    base: Path,
    config: TrainConfig,
    query_prefix: str,
    provenance: Mapping[str, Any],
    log: Log = print,
) -> Path:
    """Weights, int8 files and model card in one zip. A failed export does not lose the weights."""
    model, tokenizer = trained_model(paths, base)
    target = paths.output / "bge_small_esci"
    save_model(model, base, target)
    sample = [query_prefix + data.queries[q] for q in sorted(data.held_out)[:100]]
    original, _ = load_base(base)
    exports: dict[str, Any] = {}
    jobs = (("bge_small_esci", model), ("bge_small_base", original))
    for name, network in jobs:
        try:
            exports[name] = export_int8(
                network, tokenizer, paths.output / name, paths.work, sample, config.max_query_tokens
            )
        except Exception as error:  # an export problem must not block the download
            exports[name] = {"error": f"{type(error).__name__}: {error}"}
        log(f"{name}: {exports[name]}")
    versions = {"python": platform.python_version(), "torch": torch.__version__}
    write_results(paths, onnx=exports, provenance=dict(provenance), versions=versions)
    if paths.run_log.exists():
        write_results(paths, run=json.loads(paths.run_log.read_text(encoding="utf-8")))
    save_reference(model, tokenizer, config.max_passage_tokens, target)
    results = read_results(paths)
    (target / "training.json").write_text(json.dumps(results, indent=1) + "\n", encoding="utf-8")
    (target / "README.md").write_text(model_card(results), encoding="utf-8")
    archive = shutil.make_archive(str(paths.keep / "bge_small_esci"), "zip", paths.output)
    log(f"{archive}  {Path(archive).stat().st_size / 1e6:.0f} MB")
    return Path(archive)


def save_reference(model: Any, tokenizer: Any, limit: int, target: Path) -> None:  # noqa: ANN401
    """Vectors of fixed texts, to check that another machine reproduces this model."""
    vectors = encode(model, tokenizer, REFERENCE_TEXTS, limit, "cpu")
    body = {"texts": list(REFERENCE_TEXTS), "max_tokens": limit, "vectors": vectors.tolist()}
    path = target / "reference_embeddings.json"
    path.write_text(json.dumps(body) + "\n", encoding="utf-8")


def run_lines(results: Mapping[str, Any]) -> str:
    """Wall time and resumes, or "not logged". Never an estimate."""
    run = results.get("run")
    if not run:
        return "Wall time: not logged. Resume history: not logged."
    minutes = ", ".join(f"{s['seconds'] / 60:.1f}" for s in run["sessions"])
    resumes = "; ".join(r["message"] for r in run["resumes"]) or "none"
    return f"Training minutes per session: {minutes}. Resumes: {resumes}."


def held_out_lines(results: Mapping[str, Any]) -> str:
    """The in-domain check, with a paired interval only when per-query scores were kept."""
    before, after = results["held_out_ndcg_before"], results.get("held_out_ndcg_after")
    text = "not measured" if after is None else f"{before:.4f} before, {after:.4f} after"
    rows = [results.get(f"held_out_{x}_per_query") for x in ("before", "after")]
    if rows[0] and rows[1]:
        difference = np.array([rows[1][q] - rows[0][q] for q in sorted(rows[0])])
        pick = np.random.default_rng(20260).integers(0, len(difference), (10_000, len(difference)))
        low, high = np.percentile(difference[pick].mean(axis=1), [2.5, 97.5])
        return f"{text}. Paired difference {difference.mean():+.4f} [{low:+.4f}, {high:+.4f}]."
    return f"{text}. Per-query scores were not saved, so there is no confidence interval."


def model_card(results: Mapping[str, Any]) -> str:
    """Plain description of what was trained, on what, and what was measured."""
    data = results["data"]
    config = "\n".join(f"| {k} | {v} |" for k, v in results["config"].items())
    return f"""# bge-small-en-v1.5 fine-tuned on ESCI (US, small version)

Base: {results["provenance"]["base_model"]} at {results["provenance"]["base_revision"]}.
Code: commit {results["provenance"]["git_commit"]}.

Trained on {results["triplets"]:,} triplets from {data["train_queries"]:,} ESCI training queries
(Exact as positive, Irrelevant of the same query as hard negative for
{results["hard_negative_share"]:.0%} of them). Queries that resemble a WANDS query were removed.
No WANDS data was used.

Held-out ESCI nDCG over judged products, {data["held_out_queries"]:,} queries set aside, of which those
without an Exact product are skipped: {held_out_lines(results)}
This is an in-domain check. It says nothing about WANDS, which is measured separately.

Steps: {results["train"]["step"]:,} of {results["train"]["total"]:,}. {run_lines(results)}

| Setting | Value |
| --- | --- |
{config}

ESCI is licensed Apache-2.0 by Amazon. bge-small-en-v1.5 is licensed MIT by BAAI.
"""


In [ ]:
from scripts import finetune_dense as ft
from shelfrank.dense.finetune import TrainConfig

paths = ft.Paths(WORK, KEEP)
config = TrainConfig(**TRAINING)
data = ft.stage_data(paths, SOURCES, SETTINGS)
base = ft.download_base(WORK, MODEL["model"], MODEL["revision"])

In [ ]:
# Resumable. If it stops at the time limit or the session ends, choose Run all again.
out = ft.stage_train(paths, data, base, config, MODEL["query_prefix"], DEVICE)
print(out["step"], "of", out["total"], "steps, finished:", out["finished"])

In [ ]:
ft.stage_evaluate(paths, data, base, config, MODEL["query_prefix"], DEVICE)

In [ ]:
prefix = MODEL["query_prefix"]
archive = ft.stage_package(paths, data, base, config, prefix, PROVENANCE)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("copy this file:", archive)